# 9. Production Patterns


In [1]:
# Parameters
BATCH_MODE = "false"


## Patterns de Production : APIs Avancées OpenAI

**Navigation** : [Index](README.md) | [<< Précédent](08_Reasoning_Models.ipynb) | [Suivant >>](09b_Prompt_Security_RedTeam.ipynb)


Ce notebook couvre les fonctionnalités avancées nécessaires pour des applications en production :
- **Conversations API** : Persistance d'état entre sessions
- **Background Mode** : Tâches asynchrones longues
- **Rate Limiting** : Gestion des limites d'API
- **Optimisation** : Réduction des coûts

**Objectifs :**
- Gérer des conversations multi-sessions
- Exécuter des tâches en arrière-plan
- Implémenter des patterns de résilience
- Optimiser les coûts d'API

**Prérequis :** Notebooks 1-4

**Durée estimée :** 70 minutes

In [2]:
from pathlib import Path
%pip install -q "openai>=1.66" python-dotenv tenacity

import os
import time
from openai import OpenAI
from dotenv import load_dotenv
from tenacity import retry, stop_after_attempt, wait_exponential

# Chargement robuste de la configuration .env
from dotenv import load_dotenv
import os
# Recherche du .env dans tous les parents (pour Papermill qui change le cwd)
current_path = Path.cwd()
env_loaded = False
for _ in range(10):
    env_path = current_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path.name}")
        env_loaded = True
        break
    if current_path.name == "GenAI" or len(current_path.parts) <= 1:
        break
    current_path = current_path.parent
if not env_loaded:
    print("WARNING: .env non trouve, utilisation variables environnement")
client = OpenAI()

# Modèle par défaut depuis .env
DEFAULT_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-mini")
BATCH_MODE = os.getenv("BATCH_MODE", "false").lower() == "true"

print("Client OpenAI initialisé !")
print(f"Modèle par défaut: {DEFAULT_MODEL}")
print(f"Mode batch: {BATCH_MODE}")


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


.env charge depuis: .env


Client OpenAI initialisé !
Modèle par défaut: gpt-5-mini
Mode batch: False


### Vérification de l'Environnement

**Composants installés et chargés :**

1. **Bibliothèques Python** :
   - `openai` : SDK officiel pour l'API OpenAI
   - `python-dotenv` : Chargement sécurisé des variables d'environnement
   - `tenacity` : Gestion avancée des retry avec backoff exponentiel

2. **Configuration extraite du fichier `.env`** :
   - `OPENAI_MODEL` : Modèle par défaut (ex: `gpt-5-mini`)
   - `BATCH_MODE` : Active le mode batch pour tests automatisés (skip les inputs interactifs)

**Sortie attendue :**

```
Client OpenAI initialisé !
Modèle par défaut: gpt-5-mini
Mode batch: False
```

**Points de validation :**

| Élément | Validation | Action si erreur |
|---------|------------|------------------|
| Client initialisé | ✅ Pas d'exception | Vérifier `OPENAI_API_KEY` dans `.env` |
| Modèle détecté | ✅ Affiche un nom valide | Définir `OPENAI_MODEL` dans `.env` |
| Mode batch | ✅ `True` ou `False` | Optionnel, par défaut `False` |

> **Sécurité** : Ne JAMAIS afficher la clé API dans les logs. Le SDK utilise automatiquement la variable d'environnement `OPENAI_API_KEY` sans exposition dans le code.

## 1. Gestion du Contexte avec Chat Completions

La **gestion manuelle du contexte** avec Chat Completions permet de :
- **Maintenir l'historique** : Conserver les messages précédents dans une liste
- **Chaîner les conversations** : Chaque requête inclut tout le contexte
- **Contrôle total** : Gestion explicite de ce qui est envoyé

**Cas d'usage :**
- Conversations multi-tours
- Chatbots avec mémoire
- Systèmes de Q&A contextuels

**Architecture :**
```python
messages = [
    {"rôle": "system", "content": "..."},  # Optionnel
    {"rôle": "user", "content": "..."},
    {"rôle": "assistant", "content": "..."},  # Réponse précédente
    {"rôle": "user", "content": "..."}  # Nouvelle question
]
```

**Note :** La Responses API avec `store=True` (contexte stocké côté OpenAI) requiert `openai>=1.66` — le plancher qu'exige l'installation ci-dessus. Cette section démontre volontairement la **gestion manuelle** de l'historique avec Chat Completions : c'est le pattern le plus universel, compatible avec tout endpoint au format OpenAI.

In [3]:
# Premier message - utilisation de Chat Completions standard
# Note: la gestion du contexte est manuelle ici -- c'est l'objet de cette
# section. La Responses API avec store=True existe depuis openai>=1.66
# (le plancher exige par l'installation) ; Chat Completions reste le
# pattern universel, compatible avec tout endpoint OpenAI-compatible.

from openai import OpenAI

# Budget max_completion_tokens relevé à 4000 : gpt-5-mini est un modèle de
# raisonnement dont les tokens de raisonnement sont déduits du budget. Avec
# 200, le raisonnement pouvait consommer tout le budget sans laisser de texte
# final visible (issue #3571). 1000 laisse une marge confortable pour le contenu.

# Conversation avec historique géré manuellement
conversation_history = []

# Premier message
conversation_history.append({
    "role": "user", 
    "content": "Je m'appelle Jean et j'habite à Paris. Retiens ces informations."
})

response1 = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=conversation_history,
    max_completion_tokens=4000
)

content1 = response1.choices[0].message.content
print(f"Response 1 ID: {response1.id}")
print(f"Contenu: {content1[:200]}...")

# Ajouter la réponse à l'historique
conversation_history.append({
    "role": "assistant",
    "content": content1
})

# Deuxième message (contexte préservé via l'historique)
conversation_history.append({
    "role": "user",
    "content": "Quel est mon nom et où j'habite?"
})

response2 = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=conversation_history,
    max_completion_tokens=4000
)

content2 = response2.choices[0].message.content
print(f"\nResponse 2 ID: {response2.id}")
print(f"Contenu: {content2}")

# Vérifier les tokens
print(f"\nTokens utilisés: {response2.usage.prompt_tokens} input / {response2.usage.completion_tokens} output")

Response 1 ID: chatcmpl-EfGuQy29rpdaaJ7LTUkFiTzG0Jpxl
Contenu: D'accord — je retiens pour cette conversation que vous vous appelez Jean et que vous habitez à Paris.

Important : je peux garder ces informations pendant notre échange en cours, mais je ne peux pas l...



Response 2 ID: chatcmpl-EfGuXSmM1lkfu8JoWUR9uecued8v5
Contenu: Vous vous appelez Jean et vous habitez à Paris.

Tokens utilisés: 194 input / 85 output


### Interprétation des Résultats

**Observations importantes :**

1. **Gestion manuelle du contexte** : L'historique est conservé dans une liste `conversation_history`
2. **Contexte préservé** : En passant tout l'historique à chaque appel, le modèle a
   accès aux échanges précédents. La réponse 2 restitue correctement les
   informations mémorisées ("Vous vous appelez Jean et vous habitez à Paris."),
   preuve que le contexte est bien transmis entre les tours.
3. **Consommation de tokens** : La ligne `Tokens utilisés` de la sortie ci-dessus
   distingue les tokens d'entrée et de sortie du second appel. Comme l'entrée inclut
   tout l'historique accumulé, sa taille augmente avec les tours de conversation.

> **Note technique (issue #3571)** : `gpt-5-mini` est un modèle de raisonnement :
> les tokens de raisonnement sont déduits de `max_completion_tokens`. Le budget
> ici est relevé à 4000 pour garantir une marge confortable de contenu final
> après le raisonnement — un budget trop bas (≤500) pouvait auparavant être
> entièrement consommé par le raisonnement sans produire de texte visible.

**Pattern utilisé :**
```python
# Ajouter question utilisateur
conversation_history.append({"rôle": "user", "content": question})

# Appel API avec tout l'historique
response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=conversation_history
)

# Ajouter réponse au contexte
conversation_history.append({"rôle": "assistant", "content": response.choices[0].message.content})
```

**Optimisation :** Pour les longues conversations, envisagez de :
- Limiter l'historique aux N derniers échanges
- Résumer le contexte périodiquement
- Utiliser un système RAG pour les contextes très longs

## 2. Conversations Multi-Tours

La gestion des **conversations multi-tours** avec Chat Completions repose sur :
- **Historique cumulatif** : Chaque message s'ajoute à la liste
- **Contexte automatique** : Le modèle voit tout l'historique à chaque appel
- **Flexibilité** : Possibilité de modifier l'historique (résumé, filtrage)

**Architecture :**
```
messages = [system, user1, assistant1, user2, assistant2, user3, ...]
              ↓
        Chat Completions API
              ↓
         assistant3
```

**Avantages :**
- Simple à implémenter
- Contrôle total sur le contexte
- Compatible avec tous les modèles

In [4]:
# Simulation de conversation multi-turn avec Chat Completions
# Le contexte est géré en conservant l'historique des messages

print("=== Conversation multi-turn ===\n")

# Budget max_completion_tokens relevé à 4000 : gpt-5-mini est un modèle de
# raisonnement dont les tokens de raisonnement sont déduits du budget. Avec
# 200, le raisonnement pouvait consommer tout le budget sans texte final
# visible (issue #3571).

# Historique de conversation
messages = [
    {"role": "system", "content": "Tu es un assistant de voyage expert."},
    {"role": "user", "content": "Je planifie un voyage au Japon."}
]

# Premier échange
resp1 = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=messages,
    max_completion_tokens=4000
)
assistant_reply1 = resp1.choices[0].message.content
messages.append({"role": "assistant", "content": assistant_reply1})
print(f"Assistant: {assistant_reply1[:200]}...")

# Deuxième échange (contexte automatiquement préservé via messages)
messages.append({"role": "user", "content": "Quels sont les meilleurs endroits à Tokyo?"})
resp2 = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=messages,
    max_completion_tokens=4000
)
assistant_reply2 = resp2.choices[0].message.content
messages.append({"role": "assistant", "content": assistant_reply2})
print(f"\nAssistant: {assistant_reply2[:200]}...")

# Troisième échange
messages.append({"role": "user", "content": "Quelle est la meilleure période pour y aller?"})
resp3 = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=messages,
    max_completion_tokens=4000
)
assistant_reply3 = resp3.choices[0].message.content
messages.append({"role": "assistant", "content": assistant_reply3})
print(f"\nAssistant: {assistant_reply3[:200]}...")

print(f"\n{len([m for m in messages if m['role'] != 'system'])} messages dans la conversation")
print(f"  Contexte: Japon → Tokyo → Meilleure période")

=== Conversation multi-turn ===



Assistant: Super — je peux t’aider à tout planifier. Pour commencer, peux-tu me donner quelques précisions ?
- Durée du séjour et dates (même approximatives) ?
- Ville d’arrivée/départ prévue (Tokyo, Osaka, autr...



Assistant: Super question — Tokyo est immense et offre de tout : traditions millénaires, quartiers ultra-modernes, gastronomie de rue et musées de classe mondiale. Voici un guide synthétique des « meilleurs » en...



Assistant: Excellente question — la « meilleure » période dépend de ce que tu veux voir/faire (sakura, festivals, ski, météo calme). Voici un résumé clair pour t’aider à choisir, avec avantages/inconvénients et ...

6 messages dans la conversation
  Contexte: Japon → Tokyo → Meilleure période


### Analyse de la Conversation Multi-Tour

**Points clés observés :**

1. **Historique manuel** : Le contexte est conservé dans une liste `conversation_history` (Chat Completions), chaque tour y est ajouté avant l'appel API. Il n'y a pas de `previous_response_id` (mécanisme de la Responses API, non utilisée ici -- OpenRouter ne la supporte pas).
2. **Contexte cumulatif** : Le modèle comprend le contexte complet :
   - Message 1 : Établit le rôle (assistant de voyage) + destination (Japon)
   - Message 2 : Peut répondre spécifiquement sur Tokyo (contexte préservé)
   - Message 3 : Comprend qu'on parle toujours du Japon

3. **Structure de données** :
```
resp1 (Japon)
  └─ resp2 (Tokyo) 
      └─ resp3 (Meilleure période)
```

**Cas d'usage production :**
- Chatbots de support client (reprendre une conversation après déconnexion)
- Assistants multi-sessions (planification, conseil)
- Tutoriels interactifs (mémoriser les réponses précédentes)

### Comprendre le Polling

**Pourquoi le polling ?**

En mode background, le serveur ne peut pas "pousser" le résultat vers le client. Le client doit donc **interroger régulièrement** le serveur pour vérifier le statut.

**Pattern typique observé :**
```
t=0s   : status = "pending"     → Attendre 5s
t=5s   : status = "processing"  → Attendre 5s
t=10s  : status = "processing"  → Attendre 5s
t=15s  : status = "completed"   → Récupérer le résultat
```

**Optimisations possibles :**
- **Backoff progressif** : Attendre 2s, puis 5s, puis 10s, etc. (éviter surcharge)
- **Webhooks** : Le serveur appelle votre API quand c'est terminé (pas supporté par OpenAI actuellement)
- **WebSockets** : Connexion persistante pour notifications temps réel

**Quand utiliser background mode ?**
- ✅ Analyses longues (>30s)
- ✅ Génération de rapports complexes
- ✅ Tâches où l'utilisateur peut attendre
- ❌ Chatbot temps réel (utiliser streaming à la place)

## 3. Streaming pour UX Réactive

Le **Streaming** permet d'améliorer l'expérience utilisateur :
- **Feedback immédiat** : Les tokens apparaissent progressivement
- **Latence perçue réduite** : Premier token en ~200ms
- **Annulation possible** : L'utilisateur peut arrêter la génération

**Quand utiliser le streaming ?**
- ✅ Réponses longues (>100 tokens)
- ✅ Chatbots en temps réel
- ✅ Génération de contenu (articles, rapports)

**Alternative pour tâches longues :**
- **API Batch OpenAI** : Pour les traitements asynchrones (délai 24h)
- **Architecture async** : Celery, Redis Queue pour le polling

### Décorticage du Retry

**Fonctionnement du décorateur `@retry` :**

1. **Première tentative** : Appel normal de la fonction
2. **En cas d'échec** : Si l'exception est `RateLimitError` ou `APIError` :
   - Attendre 2s (multiplier=1, min=2)
   - Tentative 2
3. **Nouvel échec** : Attendre 4s (backoff exponentiel)
4. **Et ainsi de suite** : jusqu'à 5 tentatives max

**Exemple de scénario réel :**
```
t=0s    : Requête → RateLimitError (429 Too Many Requests)
t=2s    : Retry #1 → APIError (503 Service Unavailable)
t=6s    : Retry #2 → Succès ✅
```

**Pourquoi exponentiel ?**
- Éviter l'effet "thundering herd" : Si 1000 clients retentent simultanément après 2s, le serveur reste surchargé
- Avec backoff exponentiel, les requêtes sont étalées : 2s, 4s, 8s, 16s, 32s

**Limites :**
- ⚠️ Ne résout pas les erreurs permanentes (authentification, prompt invalide)
- ⚠️ Peut augmenter la latence totale (jusqu'à 2+4+8+16+32 = 62s)

In [5]:
# Background Mode non disponible avec Chat Completions standard
# Alternative: utiliser l'API Batch d'OpenAI pour les tâches longues

if not BATCH_MODE:
    print("=== Note sur le Background Mode ===")
    print()
    print("Le Background Mode n'est pas disponible avec l'API Chat Completions standard.")
    print()
    print("Alternatives pour les tâches longues:")
    print("  1. API Batch OpenAI: Pour les traitements asynchrones (24h délai)")
    print("  2. Streaming: Pour les réponses longues avec feedback utilisateur")
    print("  3. Architecture async: Celery, Redis Queue pour le polling")
    print()
    
    # Démonstration de streaming comme alternative
    print("=== Alternative: Streaming pour tâches longues ===")
    stream = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=[{
            "role": "user", 
            "content": "Analyse en 3 points les avantages des microservices."
        }],
        stream=True,
        # Budget aligné sur le reste du carnet (4000) : gpt-5-mini est un
        # modèle de raisonnement dont les tokens de raisonnement se déduisent
        # du budget -- à 300, deux runs consécutifs n'ont laissé aucun
        # fragment visible (même classe que #3571, cf. note section 1).
        max_completion_tokens=4000
    )
    
    # Consommer le flux réel sans fabriquer un objet de sortie par fragment.
    # Un aperçu court rafraîchit la même ligne ; la réponse entière est ensuite
    # reconstituée à partir des fragments effectivement reçus de l'API.
    print("Réponse progressive (aperçu rafraîchi pendant la réception) :")
    fragments = []
    for chunk in stream:
        texte = chunk.choices[0].delta.content
        if texte:
            fragments.append(texte)
            apercu = "".join(fragments)[-60:].replace("\n", " ")
            print(f"\r{len(fragments)} fragments | {apercu}", end="", flush=True)
    print()
    reponse_stream = "".join(fragments)
    print("Réponse complète reconstituée :")
    print(reponse_stream)
else:
    print("[BATCH_MODE] Simulation de streaming:")
    print("Réponse progressive: Les microservices offrent 3 avantages majeurs...")

=== Note sur le Background Mode ===

Le Background Mode n'est pas disponible avec l'API Chat Completions standard.

Alternatives pour les tâches longues:
  1. API Batch OpenAI: Pour les traitements asynchrones (24h délai)
  2. Streaming: Pour les réponses longues avec feedback utilisateur
  3. Architecture async: Celery, Redis Queue pour le polling

=== Alternative: Streaming pour tâches longues ===


Réponse progressive (aperçu rafraîchi pendant la réception) :
1 fragments | 1

2 fragments | 1)

3 fragments | 1) Scal

4 fragments | 1) Scalabilité

...

212 fragments |  Cette isolation simplifie aussi le diagnostic et la reprise

213 fragments |  isolation simplifie aussi le diagnostic et la reprise après

214 fragments | n simplifie aussi le diagnostic et la reprise après incident

215 fragments |  simplifie aussi le diagnostic et la reprise après incident.


Réponse complète reconstituée :
1) Scalabilité fine et optimisation des ressources  
Chaque microservice peut être mis à l’échelle indépendamment selon sa charge (par exemple monter plusieurs instances d’un service de paiement sans toucher au catalogue produit). Cela permet d’allouer CPU/mémoire de façon ciblée, réduire les coûts d’infrastructure et améliorer les performances globales.

2) Agilité, livraisons rapides et indépendance des équipes  
Les équipes peuvent développer, tester et déployer un microservice sans coordonner tous les autres, ce qui réduit les risques de régression et accélère le time-to-market. Cela facilite l’intégration continue/déploiement continu (CI/CD) et l’adoption progressive de nouvelles fonctionnalités ou technologies.

3) Résilience et isolation des pannes  
La fragmentation en services découple les défaillances : un bug ou une surcharge dans un service n’entraîne pas forcément l’arrêt de tout le système si la conception prévoit des limites (circuit brea

### Comportement du Rate Limiter

**Analyse de l'exécution :**

Dans cet exemple avec `max_requests_per_minute=10` :
- Les 3 requêtes passent **instantanément** (aucune attente)
- Pourquoi ? Parce que 3 < 10 (limite non atteinte)

**Test avec limite serrée :**

Si nous avions configuré `max_requests_per_minute=2` :
```
t=0s   : Requête 1 → OK (1/2)
t=0.5s : Requête 2 → OK (2/2)
t=1s   : Requête 3 → ATTENTE de ~59s (fenêtre de 60s)
t=60s  : Requête 3 → OK
```

**Fenêtre glissante vs Fenêtre fixe :**

Notre implémentation utilise une **fenêtre glissante** :
- ✅ Plus précis : compte exactement les 60 dernières secondes
- ✅ Pas d'effet "reset brutal" à chaque minute

Alternative **fenêtre fixe** :
```python
# Moins précis mais plus simple
if current_minute != last_minute:
    counter = 0
    last_minute = current_minute
counter += 1
if counter > max_rpm:
    wait()
```

**En production :**
- Combiner rate limiter côté client (courtoisie) + côté serveur (sécurité)
- Adapter la limite à votre tier OpenAI (voir dashboard usage limits)

## 4. Rate Limiting et Retry

En production, il est essentiel de gérer les erreurs transitoires et les limites d'API :

### Retry avec Backoff Exponentiel

La bibliothèque **tenacity** permet d'implémenter facilement un retry automatique :
- **Backoff exponentiel** : Attendre 2s, puis 4s, puis 8s, etc.
- **Retry sélectif** : Uniquement sur certaines exceptions
- **Limite de tentatives** : Éviter les boucles infinies

**Pattern recommandé :**
```python
@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=2, max=60),
    retry=retry_if_exception_type((RateLimitError, APIError))
)
```

**Erreurs à gérer :**
- `RateLimitError` : Limite de requêtes dépassée (429)
- `APIError` : Erreur serveur temporaire (500, 502, 503)
- `Timeout` : Délai d'attente dépassé

### Comparaison des Résultats

**Analyse coût/bénéfice :**

Pour ces deux requêtes similaires (résumé en 3 points), observons :

| Métrique | Test 1 (Python) | Test 2 (JavaScript) |
|----------|-----------------|---------------------|
| Tokens input | lus dans la sortie du test 1 | lus dans la sortie du test 2 |
| Tokens output | lus dans la sortie du test 1 | lus dans la sortie du test 2 |
| Coût estimé | affiché par le test 1 | affiché par le test 2 |

**Analyse des coûts :**
- `optimized_completion` est un simple Chat Completions (pas de cache activé)
- Les sorties des deux tests permettent de comparer leurs tokens et leurs coûts : une réponse plus longue coûte davantage, les autres paramètres restant identiques
- Le prompt caching OpenAI nécessite un préfixe identique répété, ce qui n'est
  pas le cas ici ("Python" vs "JavaScript")

**Test optimal pour le cache :**
```python
# Requête 1
resp1 = optimized_completion("Contexte: Documentation Python. Question: Qu'est-ce qu'une liste?")

# Requête 2 (même préfixe "Contexte: Documentation Python.")
resp2 = optimized_completion("Contexte: Documentation Python. Question: Qu'est-ce qu'un tuple?")
# → Économie attendue sur les tokens du préfixe (non mesurée dans ce carnet)
```

**Ordres de grandeur indicatifs (non mesurés dans ce carnet) :**
- Conversations longues : 40-60% économie
- RAG avec contexte fixe : 60-80% économie
- Requêtes isolées : 0% (aucun préfixe commun)

In [6]:
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from openai import RateLimitError, APIError

@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=2, max=60),
    retry=retry_if_exception_type((RateLimitError, APIError))
)
def safe_completion(prompt: str, model: str = None) -> str:
    """Appel API avec retry automatique sur erreurs transitoires"""
    model = model or DEFAULT_MODEL
    # Budget relevé à 4000 : gpt-5-mini est un modèle de raisonnement dont les
    # tokens de raisonnement sont déduits de max_completion_tokens. Avec 200,
    # le raisonnement pouvait consommer tout le budget sans texte final visible
    # (issue #3571).
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=4000
    )
    return response.choices[0].message.content

# Test
result = safe_completion("Dis 'Hello World' en 5 langues.")
print(result)

Voici "Hello World" en 5 langues :

- Français : Bonjour, le monde !  
- Espagnol : ¡Hola, mundo!  
- Allemand : Hallo, Welt!  
- Italien : Ciao, mondo!  
- Japonais : こんにちは世界 (Konnichiwa sekai)


### Interprétation des Logs

**Format de log structuré :**

```
2026-02-04 15:23:45 | OpenAI_Production | INFO | SUCCESS | Model: gpt-5-mini | Duration: 1.23s | Tokens: 85 (in: 25, out: 60)
```

**Champs clés :**
- `Timestamp` : Horodatage précis (important pour corrélation)
- `Level` : INFO (succès) ou ERROR (échec)
- `Model` : Tracer quel modèle a été utilisé
- `Duration` : Latence end-to-end (objectif : <2s pour chatbot)
- `Tokens` : Consommation (surveiller les dépassements)

**Analyse des logs :**

1. **Log SUCCESS** :
   - Durée ~1-2s : Normal pour gpt-5-mini
   - 85 tokens total : Petit prompt + réponse courte
   - ✅ Requête efficace

2. **Log ERROR** :
   - `InvalidRequestError` : Modèle invalide
   - Capturé et loggé avant propagation
   - ✅ Permet diagnostic rapide sans crash

**Agrégation recommandée :**
```bash
# Latence moyenne par modèle (via grep + awk)
grep "SUCCESS" production.log | awk '{print $8, $12}' | sort

# Taux d'erreur sur 1h
grep "ERROR" production.log | wc -l
```

**Outils professionnels :**
- ELK Stack (Elasticsearch, Logstash, Kibana)
- Datadog, Splunk, New Relic
- Prometheus + Grafana

### Transition vers les Patterns Avancés

Jusqu'ici nous avons couvert les **patterns de résilience et d'optimisation** :
- ✅ Retry automatique
- ✅ Rate limiting
- ✅ Logging structuré
- ✅ Optimisation des coûts

Les deux derniers patterns concernent **l'expérience utilisateur et la sécurité** :

1. **Streaming** : Améliorer la perception de latence
   - Au lieu d'attendre 5s pour afficher 500 tokens
   - Afficher progressivement (premier token en 200ms)
   - Utilisateur voit la "pensée en temps réel"

2. **Modération** : Filtrer le contenu inapproprié
   - Avant de traiter : vérifier que l'input est acceptable
   - Après génération : s'assurer que l'output est sûr
   - Éviter les violations de politique d'usage

Ces patterns sont **essentiels** pour toute application destinée aux utilisateurs finaux.

### Analyse des Résultats

**Streaming :**

Observez le comportement :
- Les tokens apparaissent **progressivement** (un par un ou par petits groupes)
- Latence au premier token : ~200-500ms
- Latence totale : Identique à une requête non-streamée (~2s)
- **Gain perçu** : L'utilisateur voit le début de la réponse immédiatement

**Implémentation UX :**
```python
# Dans une vraie app web
async def stream_to_user():
    for chunk in stream:
        content = chunk.choices[0].delta.content
        if content:
            await websocket.send(content)  # Envoyer au navigateur en temps réel
```

**Modération :**

Les résultats montrent :
1. **Texte neutre** : "Bonjour, comment vas-tu?"
   - `flagged = False` : Aucun problème détecté
   
2. **Texte menaçant** : "Enfoiré, je vais te tuer!"
   - `flagged = True` : Détecté (harcèlement, menace, violence)

**Cas qui seraient flaggés :**
- Contenu haineux, violent, sexuel
- Harcèlement, menaces
- Auto-mutilation

**Pattern de sécurité :**
```python
# Avant de traiter
if moderate(user_input).flagged:
    return "Désolé, ce contenu viole notre politique."

# Générer
response = optimized_completion(user_input)["content"]

# Après génération
if moderate(response).flagged:
    return "Désolé, impossible de répondre à cette demande."
```

### Rate Limiter Personnalisé

Pour respecter les limites de requêtes par minute (RPM), implémentons un **rate limiter** :
- **Fenêtre glissante** : Compte les requêtes sur les 60 dernières secondes
- **Attente automatique** : Bloque jusqu'à ce qu'une requête soit autorisée
- **Configurable** : Adapter selon votre tier OpenAI

**Limites par tier (exemple) :**
- Free : 3 RPM
- Tier 1 : 500 RPM
- Tier 2 : 5000 RPM
- Tier 5 : 10000 RPM

In [7]:
import time
from collections import deque

class RateLimiter:
    """Limite le nombre de requêtes par minute"""
    def __init__(self, max_requests_per_minute: int = 60):
        self.max_rpm = max_requests_per_minute
        self.requests = deque()
    
    def wait_if_needed(self):
        now = time.time()
        # Nettoyer les anciennes requêtes (plus de 60s)
        while self.requests and now - self.requests[0] > 60:
            self.requests.popleft()
        
        # Si limite atteinte, attendre
        if len(self.requests) >= self.max_rpm:
            wait_time = 60 - (now - self.requests[0])
            if wait_time > 0:
                print(f"Rate limit: attente de {wait_time:.1f}s")
                time.sleep(wait_time)
        
        self.requests.append(time.time())

# Exemple d'utilisation
limiter = RateLimiter(max_requests_per_minute=10)

for i in range(3):
    limiter.wait_if_needed()
    result = safe_completion(f"Nombre aléatoire #{i+1}")
    print(f"Requête {i+1}: {result[:50]}...")

Requête 1: Voici un nombre aléatoire (entier entre 1 et 100) ...


Requête 2: Voulez-vous un nombre entier ou un réel, et dans q...


Requête 3: Voulez-vous un entier ou un nombre décimal, et dan...


### Exemple guidé 1 (crédité #21043) : Retry decorateur avec logging

L'objectif est de créer un decorateur `@retry_with_log` qui wrappe n'importe quel appel API avec retry automatique ET logging structure de chaque tentative.

**Indices :**
- `# Étape 1` : Définir le decorateur avec paramètres (max_retries, backoff_base)
- `# Étape 2` : Dans le wrapper, boucler sur les tentatives avec try/except
- `# Étape 3` : Logger chaque tentative (numéro, duree, succes/echec)
- `# Étape 4` : Utiliser time.sleep avec un delai exponentiel entre les tentatives

> **Exemple guidé (crédité #21043).** La résolution ci-dessous est celle du groupe de la PR #21043, conservée telle quelle : décorateur à paramètres validés, backoff exponentiel et journal de chaque tentative. La docstring signale elle-même la limite de production (toutes les exceptions sont retentées) — l'exercice 1bis qui la suit corrige précisément ce point avec un filtre d'erreurs transitoires.


In [8]:
import time
import math
import logging
from functools import wraps
def retry_with_log(max_retries=3, backoff_base=2):
    """max_retries = reprises après la première tentative (donc 4 appels par défaut).

    Cet exercice retente Exception. En production, filtrer les erreurs transitoires
    et vérifier l'idempotence afin d'éviter de facturer deux fois une opération.
    """
    if isinstance(max_retries, bool) or not isinstance(max_retries, int) or max_retries < 0:
        raise ValueError('max_retries doit être un entier positif ou nul')
    if isinstance(backoff_base, bool) or not isinstance(backoff_base, (int, float)) or not math.isfinite(backoff_base) or backoff_base <= 0:
        raise ValueError('backoff_base doit être fini et strictement positif')
    def decorate(function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            logger = logging.getLogger('RetryTP')
            for attempt in range(max_retries + 1):
                start = time.perf_counter()
                try:
                    result = function(*args, **kwargs)
                except Exception as error:
                    final = attempt == max_retries
                    delay = 0 if final else backoff_base ** attempt
                    logger.warning('timestamp=%s tentative=%d durée=%.6f succès=False exception=%s délai=%s',
                                   time.time(), attempt + 1, time.perf_counter() - start, type(error).__name__, delay)
                    if final:
                        raise
                    time.sleep(delay)
                else:
                    logger.info('timestamp=%s tentative=%d durée=%.6f succès=True',
                                time.time(), attempt + 1, time.perf_counter() - start)
                    return result
        return wrapper
    return decorate


from unittest.mock import patch
_attempts = []
@retry_with_log(max_retries=2)
def _test_retry():
    _attempts.append(1)
    if len(_attempts) < 3:
        raise TimeoutError('Erreur simulée')
    return 'Réussite'
with patch.object(time, 'sleep') as _sleep:
    print(_test_retry())
    print('Nombre de tentatives :', len(_attempts))
    print('Attentes simulées :', [c.args[0] for c in _sleep.call_args_list])


timestamp=1793505225.3870406 tentative=1 durée=0.000004 succès=False exception=TimeoutError délai=1


timestamp=1793505225.388044 tentative=2 durée=0.000007 succès=False exception=TimeoutError délai=2


Réussite
Nombre de tentatives : 3
Attentes simulées : [1, 2]


### Exercice 1bis - Réessais sélectifs : ne retenter que le transitoire

L'exemple guidé 1 retente **toutes** les exceptions — sa propre docstring le signale comme une limite de production. En pratique, un `ValueError` de validation ne se corrige pas en attendant : le retenter facture des appels pour rien.

**Objectif** : construire `retry_transient(function, retries=3, transient=(ConnectionError, TimeoutError))` qui ne retente que les exceptions du tuple `transient` et laisse remonter immédiatement les autres.

**Indices** :
- `# Étape 1` : écrire deux fonctions de test — `flaky()` qui lève `ConnectionError` deux fois puis réussit, et `invalid()` qui lève `ValueError` dès le premier appel (comptez les appels avec un compteur par fonction)
- `# Étape 2` : appeler les deux via `retry_transient` et capturer l'exception de `invalid` avec `try/except`
- `# Étape 3` : relever `appels_flaky` (doit atteindre 3) et `appels_invalid` (doit rester à 1 — l'échec non transitoire ne retente pas)
- `# Indice` : le filtre se fait sur `isinstance(error, transient)` dans le `except` ; attention à ne pas attraper `BaseException`

In [9]:
# Exercice 1bis - Réessais sélectifs : ne retenter que le transitoire
# TODO etudiant : retenter uniquement les exceptions transitoires
# Etape 1 : deux fonctions de test, flaky (2 ConnectionError puis succes) et
#           invalid (ValueError immediat), chacune comptant ses appels
# Etape 2 : definir retry_transient(function, retries=3, transient=(...))
#           qui relaisse remonter les exceptions non transitoires
# Etape 3 : relever appels_flaky, appels_invalid et l'exception remontee
# Indice : isinstance(error, transient) dans le except ; jamais BaseException
appels_flaky = None  # TODO etudiant
appels_invalid = None  # TODO etudiant
exception_remontee = None  # TODO etudiant
print("Exercice 1bis a completer : réessais sélectifs")

Exercice 1bis a completer : réessais sélectifs


### Exemple guidé 1b (crédité #22362) : la même consigne, une autre plume

La consigne est celle de l'exemple guidé 1 ; la solution ci-dessous vient d'un **second groupe d'étudiants**, conservée quasi verbatim — seule la première ligne est retitrée, et l'auto-test, qui vivait dans la cellule de l'exercice 2 de la livraison d'origine, est déplacé ici (écarts déclarés dans la PR d'intégration).

La lecture la plus instructive est la comparaison des deux versions : même consigne, même décorateur, **deux sémantiques de `max_retries`**. Ici, `max_retries` compte les tentatives **au total** (3 par défaut = 3 appels, donc 2 attentes) ; l'exemple guidé 1 compte les **reprises après la première tentative** (3 par défaut = 4 appels, donc 3 attentes). Les deux lectures sont fidèles à l'énoncé — et ce groupe épinglera la sienne dans sa docstring, là où l'ambiguïté se niche.

In [10]:
# Exemple guide 1b (credite #22362) : Retry decorateur avec logging
# Consigne d'origine, conservee comme contexte de cette resolution guidee :
# implementer @retry_with_log avec max_retries=3 et backoff_base=2 par defaut ;
# intercepter les exceptions et retenter ; journaliser chaque tentative avec son
# horodatage, son numero, le type d'exception et le delai avant l'essai suivant ;
# retourner le resultat en cas de succes et relever la derniere exception sinon.
# La solution complete et son auto-test suivent ce rappel de la consigne.

import functools
import time
from datetime import datetime


def retry_with_log(max_retries=3, backoff_base=2):
    """
    Decorateur : retente automatiquement la fonction en cas d'exception.
    max_retries : nombre total de tentatives (defaut 3)
    backoff_base : base du delai exponentiel (delai = backoff_base ** n_tentative
    -> 2s, 4s, 8s... avec la base 2)
    Log chaque tentative : timestamp, n_tentative, duree, type d'exception, delai
    Retourne le resultat en cas de succes, leve la derniere exception si echec total
    """
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            for attempt in range(1, max_retries + 1):
                start = time.time()
                try:
                    result = func(*args, **kwargs)
                    duration = time.time() - start
                    print(f"{datetime.now():%Y-%m-%d %H:%M:%S} | {func.__name__} | "
                          f"Tentative {attempt}/{max_retries} | SUCCES | "
                          f"Duree: {duration:.2f}s")
                    return result
                except Exception as e:
                    duration = time.time() - start
                    last_exception = e
                    ts = f"{datetime.now():%Y-%m-%d %H:%M:%S}"
                    if attempt < max_retries:
                        delay = backoff_base ** attempt
                        print(f"{ts} | {func.__name__} | "
                              f"Tentative {attempt}/{max_retries} | ECHEC | "
                              f"Duree: {duration:.2f}s | {type(e).__name__}: {str(e)[:80]} | "
                              f"Nouvel essai dans {delay}s")
                        time.sleep(delay)
                    else:
                        print(f"{ts} | {func.__name__} | "
                              f"Tentative {attempt}/{max_retries} | ECHEC DEFINITIF | "
                              f"Duree: {duration:.2f}s | {type(e).__name__}: {str(e)[:80]}")
            raise last_exception
        return wrapper
    return decorator


# Variante de test avec l'API, laissee en commentaire : sans cle OpenAI,
# l'appel reel echouerait. L'auto-test sans API ci-dessous rejoue le decorateur
# sur des fonctions locales.
# @retry_with_log(max_retries=3, backoff_base=2)
# def call_api(prompt):
#     return safe_completion(prompt)
#
# resultat = call_api("Test retry decorateur")
# print(resultat)


# Auto-test local du decorateur : il n'exerce que les reprises, sans appel API.

if __name__ == "__main__":
    print("--- Auto-test de l exemple guide 1b ---")
    calls = {"n": 0}

    @retry_with_log(max_retries=3, backoff_base=1)  # base 1 -> delais courts pour le test
    def flaky():
        calls["n"] += 1
        if calls["n"] < 3:
            raise ConnectionError("erreur transitoire simulee")
        return "OK"

    print("Resultat:", flaky())

    @retry_with_log(max_retries=2, backoff_base=1)
    def always_fails():
        raise ValueError("erreur permanente")

    try:
        always_fails()
    except ValueError as e:
        print("Exception finale levee:", e)


--- Auto-test de l exemple guide 1b ---
2026-11-01 04:53:45 | flaky | Tentative 1/3 | ECHEC | Duree: 0.00s | ConnectionError: erreur transitoire simulee | Nouvel essai dans 1s


2026-11-01 04:53:46 | flaky | Tentative 2/3 | ECHEC | Duree: 0.00s | ConnectionError: erreur transitoire simulee | Nouvel essai dans 1s


2026-11-01 04:53:47 | flaky | Tentative 3/3 | SUCCES | Duree: 0.00s
Resultat: OK
2026-11-01 04:53:47 | always_fails | Tentative 1/2 | ECHEC | Duree: 0.00s | ValueError: erreur permanente | Nouvel essai dans 1s


2026-11-01 04:53:48 | always_fails | Tentative 2/2 | ECHEC DEFINITIF | Duree: 0.00s | ValueError: erreur permanente
Exception finale levee: erreur permanente


### Lecture du résultat — exemple guidé 1b

Le journal ci-dessus est celui de **cette exécution**. On y lit les deux scénarios côte à côte :

- `flaky` échoue aux tentatives 1 et 2, réussit à la 3 : chaque échec affiche l'heure, le type d'exception (`ConnectionError`), la durée et le **délai annoncé avant l'essai suivant** — ici 1 s, puis 1 s (`backoff_base=1`, donc `1**n`), choisi pour ne pas ralentir le test.
- `always_fails` échoue une première fois, réessaie 1 s plus tard, et s'arrête sur `ECHEC DEFINITIF` à la tentative 2/2 ; l'exception d'origine (`ValueError: erreur permanente`) est **relevée telle quelle** et récupérée par l'appelant : c'est bien la *dernière* exception, pas une exception générique qui masquerait sa cause.

**Comparer avec l'exemple guidé 1.** Ici `flaky` réussit en exactement **3 appels** avec `max_retries=3` ; le même scénario dans l'exemple 1 aurait toléré un échec de plus, car `max_retries` y désigne les **reprises après la première tentative**. Les deux lectures de la consigne cohabitent dans ce carnet : la valeur du groupe est donc surtout dans la docstring, qui **énonce explicitement la sémantique choisie** (« nombre total de tentatives ») — c'est ce qu'un appelant lit avant de brancher son appel facturé dessus.

Deux limites du groupe, conservées telles quelles dans le code et à corriger en exercice : le message de la docstring `2s, 4s, 8s` décrit un backoff `base**n` à partir de 0, alors que la boucle commence à 1 (`base**1 = 2` pour la base 2) ; et `raise last_exception` devient `raise None` (un `TypeError`) si `max_retries=0`. L'exercice **1ter** part précisément de ce cas limite : un budget de temps implique de renoncer *proprement*, y compris avant toute tentative.

### Exercice 1ter - Retry budgeté : renoncer à temps

Le décorateur ci-dessus retente jusqu'à épuisement des tentatives, même quand la somme des délais dépasse toute attente raisonnable. Écrire `retry_with_budget(max_retries=5, budget_seconds=3)` qui :

- retente avec le même backoff exponentiel, mais **renonce avant** toute tentative dont le délai annoncé ne tiendrait plus dans le budget restant (temps écoulé + délai > budget) ;
- lève alors la dernière exception en précisant dans le message `budget épuisé après N tentatives` ;
- ne lève jamais `None` : vérifier que `max_retries=0` lève bien l'exception de la fonction, et non un `TypeError` — le piège de la variante ci-dessus.

**Indices :**
- `# Étape 1` : repartir du squelette de `retry_with_log` (décorateur paramétré)
- `# Étape 2` : mesurer le budget restant avec `time.monotonic()` (l'horloge murale utilisée ci-dessus peut reculer)
- `# Étape 3` : avant chaque tentative, comparer délai annoncé et budget restant ; renoncer proprement sinon
- `# Étape 4` : tester `max_retries=0` sur une fonction qui échoue toujours

In [11]:
# Exercice 1ter - Retry budgete : renoncer a temps
# TODO etudiant
# Etape 1 : reproduire le squelette de retry_with_log (decorateur parametre)
# Etape 2 : mesurer le temps restant avec time.monotonic() avant chaque tentative
# Etape 3 : si le delai annonce ne tient plus dans le budget, arreter et lever
#           la derniere exception avec un message explicite
# Etape 4 : verifier que max_retries=0 leve l'exception de la fonction (pas None)

import functools
import time


def retry_with_budget(max_retries=5, budget_seconds=3.0, backoff_base=2):
    """Retente la fonction dans un budget de temps total ; renonce avant la
    tentative dont le delai annonce ne tiendrait plus dans le budget restant."""
    pass  # TODO etudiant


# Auto-test attendu (a decommenter une fois complete) :
# @retry_with_budget(max_retries=5, budget_seconds=3, backoff_base=2)
# def never_succeeds():
#     raise ConnectionError("panne durable")
#
# try:
#     never_succeeds()
# except ConnectionError as e:
#     print("Levee dans le budget:", e)

print("Exercice a completer")

Exercice a completer


### Interprétation du Comportement du Rate Limiter

**Observations de l'exécution :**

Les 3 requêtes se sont exécutées **sans attente** car la limite (`max_requests_per_minute=10`) n'a pas été atteinte.

**Analyse du mécanisme :**

1. **Fenêtre glissante** :
   ```python
   # À t=0s : deque([t0])         → 1/10, OK
   # À t=1s : deque([t0, t1])     → 2/10, OK  
   # À t=2s : deque([t0, t1, t2]) → 3/10, OK
   ```

2. **Nettoyage automatique** :
   - Les timestamps > 60s sont supprimés du deque
   - Garantit que seules les requêtes récentes comptent

**Test avec limite stricte :**

Si nous avions configuré `max_requests_per_minute=2` :

```
t=0.0s : Requête #1 → OK (1/2)
t=0.5s : Requête #2 → OK (2/2)
t=1.0s : Requête #3 → ATTENTE ~59s (limite atteinte)
         ↓ Nettoyage à t=60s (requête #1 expire)
t=60.5s: Requête #3 → OK (1/2)
```

**Comparaison des stratégies :**

| Approche | Avantages | Inconvénients |
|----------|-----------|---------------|
| **Fenêtre glissante** (notre impl.) | ✅ Précis, pas d'effet de bord | ⚠️ Mémoire O(n) requêtes |
| **Fenêtre fixe** (reset à chaque minute) | ✅ Simple, O(1) mémoire | ⚠️ Burst de 2× limite possible |
| **Token bucket** | ✅ Permet les bursts contrôlés | ⚠️ Plus complexe à implémenter |

**Recommandations production :**

- **Développement** : Fenêtre glissante (notre code)
- **Production haute charge** : Token bucket avec Redis
- **Edge cases** : Combiner rate limiter client + serveur (double protection)

> **Note** : Les limites OpenAI sont par organisation, pas par application. Coordonner le rate limiting si plusieurs services utilisent la même clé API.

## 5. Optimisation des Coûts

Stratégies pour réduire les coûts d'API en production :

### 1. Utiliser le Cache (store=True)
- Économie de 40-80% sur les tokens d'entrée répétés
- Activer sur toutes les conversations longues

### 2. Choisir le Bon Modèle
| Modèle | Prix Input | Prix Output | Cas d'usage |
|--------|------------|-------------|-------------|
| gpt-5-mini | $0.15/1M | $0.60/1M | Tâches simples, production |
| gpt-5 | $2.50/1M | $10.00/1M | Tâches complexes |
| o1 | $10.00/1M | $40.00/1M | Raisonnement profond |

### 3. Limiter les Tokens
- Utiliser `max_tokens` pour contrôler la longueur
- Préférer les prompts courts et précis
- Éviter les contextes inutilement longs

### 4. Batch Processing
- Utiliser l'API Batch pour réductions de 50%
- Acceptable pour tâches non-temps-réel

### 5. Monitoring et Alertes
- Suivre les coûts quotidiens/mensuels
- Configurer des alertes budgétaires

In [12]:
def optimized_completion(prompt: str, system_prompt: str = None) -> dict:
    """Completion optimisée avec métriques de coût"""
    
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    
    # Budget relevé à 4000 : gpt-5-mini est un modèle de raisonnement dont les
    # tokens de raisonnement sont déduits de max_completion_tokens. Avec 200,
    # le raisonnement pouvait consommer tout le budget sans texte final visible
    # (issue #3571).
    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=messages,
        max_completion_tokens=4000
    )
    
    # Calculer les coûts approximatifs
    # Prix gpt-5-mini: ~$0.15/1M input, ~$0.60/1M output
    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens
    
    cost_input = input_tokens * 0.00000015
    cost_output = output_tokens * 0.0000006
    
    return {
        "content": response.choices[0].message.content,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "estimated_cost": cost_input + cost_output
    }

# Test 1
result1 = optimized_completion("Résume les avantages de Python en 3 points.")
print("=== Test 1 ===")
print(f"Réponse: {result1['content'][:100]}...")
print(f"Tokens: {result1['input_tokens']} in / {result1['output_tokens']} out")
print(f"Coût estimé: ${result1['estimated_cost']:.6f}")

# Test 2
result2 = optimized_completion("Résume les avantages de JavaScript en 3 points.")
print("\n=== Test 2 ===")
print(f"Réponse: {result2['content'][:100]}...")
print(f"Tokens: {result2['input_tokens']} in / {result2['output_tokens']} out")
print(f"Coût estimé: ${result2['estimated_cost']:.6f}")

=== Test 1 ===
Réponse: - Syntaxe claire et lisible : code concis et expressif, facile à apprendre et à maintenir (indentati...
Tokens: 17 in / 384 out
Coût estimé: $0.000233



=== Test 2 ===
Réponse: 1) Universalité — JavaScript s’exécute dans tous les navigateurs et, via Node.js, côté serveur (ains...
Tokens: 18 in / 539 out
Coût estimé: $0.000326


### Exemple guidé 2 (crédité #21043) : Cache simple pour prompts repetes

L'objectif est d'implementer un cache memoire qui stocke les résultats des appels API pour eviter de re-appeler le modèle sur des prompts identiques.

**Indices :**
- `# Étape 1` : Créer une classe SimpleCache avec un dictionnaire interne
- `# Étape 2` : Implementer get(key) qui retourne la valeur ou None si absente
- `# Étape 3` : Implementer set(key, value, ttl_seconds) avec un timestamp d'expiration
- `# Étape 4` : Implementer cached_completion(prompt) qui utilise le cache avant d'appeler l'API

> **Exemple guidé (crédité #21043).** La résolution ci-dessous est celle du groupe de la PR #21043, conservée telle quelle : cache TTL sur horloge monotone injectable, sentinel `None` pour l'absence, injection de `completion_fn` pour les tests hors ligne. L'exercice 2bis qui la suit mesure, lui, le taux de hit réel d'une séquence de prompts avec expiration.


In [13]:
import time
import math
import logging
from functools import wraps
class SimpleCache:
    """TTL en secondes, horloge monotone. None indique une entrée absente."""
    def __init__(self, clock=None):
        self._clock = clock or time.monotonic
        self._data = {}

    def get(self, key):
        entry = self._data.get(key)
        if entry is None:
            return None
        value, expiration = entry
        if self._clock() >= expiration:
            del self._data[key]
            return None
        return value

    def set(self, key, value, ttl_seconds=300):
        if isinstance(ttl_seconds, bool) or not isinstance(ttl_seconds, (int, float)) or not math.isfinite(ttl_seconds) or ttl_seconds < 0:
            raise ValueError('TTL invalide')
        if value is None:
            raise ValueError('None est réservé à une absence de cache')
        self._data[key] = (value, self._clock() + ttl_seconds)

cache = SimpleCache()

def cached_completion(prompt, completion_fn=None, ttl_seconds=300):
    """Dans le notebook, utilise safe_completion. Injection pour les tests hors ligne.

    Clé = prompt exact. Si modèle/options changent, les inclure dans la clé ou vider
    ce cache. Les réponses en erreur ne sont jamais stockées.
    """
    result = cache.get(prompt)
    if result is not None:
        return result
    if completion_fn is None:
        completion_fn = globals().get('safe_completion')
    if completion_fn is None:
        raise RuntimeError('Fournir completion_fn ou définir safe_completion')
    result = completion_fn(prompt)
    cache.set(prompt, result, ttl_seconds)
    return result


_cache_calls=[]
def _fake_completion(prompt):
    _cache_calls.append(prompt)
    return 'Réponse simulée'
print(cached_completion('Question', _fake_completion))
print(cached_completion('Question', _fake_completion))
print('Appels au fournisseur simulé :', len(_cache_calls))


Réponse simulée
Réponse simulée
Appels au fournisseur simulé : 1


### Exercice 2bis - Le taux de hit réel : mesurer ce que le cache économise

L'exemple guidé 2 démontre `SimpleCache` sur un hit et un miss isolés. Ce qui intéresse l'exploitation, c'est la **séquence** : sur une charge réelle, quelle fraction des appels le cache absorbe-t-il ?

**Objectif** : mesurer hits, misses et taux de hit d'une séquence donnée, puis vérifier l'expiration TTL en avançant l'horloge.

**Indices** :
- `# Étape 1` : construire un `SimpleCache(clock=...)` avec une horloge factice (classe dont `monotonic()` retourne une valeur que vous incrémentez à la main) et un `completion_fn` qui compte les appels réels
- `# Étape 2` : rejouer la séquence `["p1", "p2", "p1", "p3", "p1", "p2"]` via `cached_completion`, relever `hits` et `misses` (les misses = appels réels du compteur)
- `# Étape 3` : calculer `taux_hit = hits / 6` ; avancer ensuite l'horloge au-delà du TTL et vérifier que le même prompt redevient un miss (`expiration_observee`)
- `# Indice` : `hits + misses = 6` ; si votre total dépasse, vous comptez un appel réel comme un hit

In [14]:
# Exercice 2bis - Le taux de hit reel : mesurer ce que le cache economise
# TODO etudiant : mesurer hits/misses d'une sequence puis l'expiration TTL
# Etape 1 : SimpleCache avec horloge factice + completion_fn compteur
# Etape 2 : rejouer la sequence p1,p2,p1,p3,p1,p2 via cached_completion
# Etape 3 : taux_hit = hits/6 ; avancer l'horloge past TTL et observer
# Indice : hits + misses = 6 ; les appels reels du compteur = les misses
hits = None  # TODO etudiant
misses = None  # TODO etudiant
taux_hit = None  # TODO etudiant
expiration_observee = None  # TODO etudiant
print("Exercice 2bis a completer : taux de hit du cache")

Exercice 2bis a completer : taux de hit du cache


### Exemple guidé 2b (crédité #22362) : le même cache, une autre main

Deuxième solution créditée pour la même consigne que l'exemple guidé 2, quasi verbatim. Ce groupe ajoute un **compteur de hits/misses directement dans le cache** (`stats()`), expire les entrées paresseusement au `get`, et signale chaque échange réseau par un simple `print`.

Là où l'exemple guidé 2 rend la fonction de completion **injectable** (`completion_fn`) pour rester testable hors ligne, cette version appelle le `safe_completion` global du carnet — et son auto-test le stubait en **rebindant le nom global**, ce qui changerait le comportement des cellules suivantes : le stub est donc renommé et le global sauvegardé puis restauré autour des appels de test (écart déclaré).

In [15]:
# Exemple guide 2b (credite #22362) : Cache simple pour prompts repetes
# Consigne d'origine, conservee comme contexte de cette resolution guidee :
# implementer un cache memoire optimise par SimpleCache ; stocker pour chaque cle
# le couple (valeur, horodatage d'expiration) ; get rend la valeur presente et non
# expiree, sinon None ; set accepte un TTL de 300 secondes par defaut ;
# cached_completion consulte le cache avant d'appeler safe_completion, puis stocke
# la nouvelle reponse ; la cle utilisee ici est hash(prompt).
# La solution complete et son auto-test suivent ce rappel de la consigne.


class SimpleCache:
    """Cache memoire avec expiration (TTL) : {cle: (valeur, timestamp_expiration)}"""

    def __init__(self):
        self._store = {}
        self.hits = 0
        self.misses = 0

    def get(self, key):
        """Retourne la valeur si presente et non expiree, sinon None."""
        entry = self._store.get(key)
        if entry is None:
            self.misses += 1
            return None
        value, expiry = entry
        if time.time() > expiry:
            del self._store[key]  # entree expiree -> on la supprime
            self.misses += 1
            return None
        self.hits += 1
        return value

    def set(self, key, value, ttl_seconds=300):
        """Stocke la valeur avec un TTL (defaut : 5 minutes)."""
        self._store[key] = (value, time.time() + ttl_seconds)

    def stats(self):
        total = self.hits + self.misses
        rate = (self.hits / total * 100) if total else 0
        return {"hits": self.hits, "misses": self.misses, "hit_rate_%": round(rate, 1)}


cache = SimpleCache()


def cached_completion(prompt: str, ttl_seconds: int = 300) -> str:
    """Check le cache ; si miss -> appelle safe_completion -> stocke -> retourne."""
    key = hash(prompt)  # suffisant pour un cache en memoire (le hash change entre sessions Python)
    cached = cache.get(key)
    if cached is not None:
        print("[CACHE HIT]")
        return cached
    print("[CACHE MISS] appel API...")
    result = safe_completion(prompt)  # definie plus haut dans le notebook
    cache.set(key, result, ttl_seconds)
    return result


# Test (avec l'API) -- laisse en commentaire : sans cle OpenAI, l'appel reel
# echouerait ; le bloc d'auto-test ci-dessous rejoue les deux appels avec un stub.
# r1 = cached_completion("Quelle est la capitale de la France?")
# r2 = cached_completion("Quelle est la capitale de la France?")  # utilise le cache
# print(f"Cache hit? {r1 == r2}")
# print(cache.stats())

# Auto-test local du cache. Le fournisseur simule porte un nom distinct de
# safe_completion ; le nom global du carnet est sauvegarde puis restaure autour
# des appels de test pour ne pas modifier les exemples suivants.

if __name__ == "__main__":
    print("--- Auto-test de l exemple guide 2b ---")

    def _safe_completion_stub(prompt):  # renomme depuis safe_completion
        return f"reponse a: {prompt}"

    _api_completion = safe_completion
    safe_completion = _safe_completion_stub
    try:
        r1 = cached_completion("Quelle est la capitale de la France?")
        r2 = cached_completion("Quelle est la capitale de la France?")
    finally:
        safe_completion = _api_completion
    print("Cache hit?", r1 == r2, cache.stats())

    cache.set("k", "v", ttl_seconds=1)
    time.sleep(1.1)
    print("Apres expiration:", cache.get("k"))


--- Auto-test de l exemple guide 2b ---
[CACHE MISS] appel API...
[CACHE HIT]
Cache hit? True {'hits': 1, 'misses': 1, 'hit_rate_%': 50.0}


Apres expiration: None


### Lecture du résultat — exemple guidé 2b

Sorties de cette exécution : le premier appel imprime `[CACHE MISS] appel API...`, le second — **même prompt** — imprime `[CACHE HIT]` ; la vérification `r1 == r2` vaut `True` et `stats()` affiche `{'hits': 1, 'misses': 1, 'hit_rate_%': 50.0}`. Enfin, après `set("k", "v", ttl_seconds=1)` puis une attente de 1,1 s, `get("k")` rend `None` : l'entrée expirée a été supprimée au passage du `get`.

Le taux de hit de 50 % n'est **pas** une mesure d'efficacité : c'est le compte d'un test de deux appels identiques. Ce qu'il démontre, c'est que le deuxième appel n'a **pas** déclenché de réseau (le stub local, substitué à `safe_completion` le temps du test, n'est appelé qu'une fois) et que l'expiration est réelle. Pour mesurer ce qu'un cache **économise** sur un trafic réel, voir l'exercice 2bis ci-dessus, qui fait varier la distribution des prompts.

Deux différences de conception avec l'exemple guidé 2 méritent d'être nommées plutôt que choisies d'office :

- **Expiration paresseuse vs horloge** : cette version utilise `time.time()` (horloge murale) pour le TTL ; l'exemple guidé 2 prend `time.monotonic()` — le seul choix qui garantisse qu'une entrée ne « dé-expire » pas si l'horloge système recule.
- **Sentinelle `None`** : `get` rend `None` pour « absent ou expiré », et `cached_completion` traite donc une valeur `None` stockée comme un miss (elle serait re-facturée à chaque appel). Ici c'est bénin — `safe_completion` retourne une chaîne — mais la distinction *présent-avec-valeur-None* / *absent* devient nécessaire dès que le résultat mis en cache peut être `None`.

Et la limite structurelle, qui fait l'objet de l'exercice **2ter** : la clé est `hash(prompt)` seul. Elle ignore le modèle, la température, le message système — tout ce qui, à prompt identique, change la réponse. Un cache de production met ces paramètres **dans la clé**.


### Exercice 2ter - Faire échouer le cache

Le cache ci-dessus semble inoffensif : il ne fait que répondre plus vite. Lui faire servir une **mauvaise réponse** :

1. **Clé amnésique** : `cached_completion` ne connaît que le prompt. Écrire une `cached_completion_v2(prompt, model)` naïve (même clé de cache), l'appeler avec le même prompt et deux « modèles » simulés par deux stubs qui répondent différemment — montrer la mauvaise réponse servie depuis le cache.
2. **Collision** : remplacer la clé par les 8 premiers caractères du prompt (`key = prompt[:8]`), puis construire deux prompts distincts qui entrent en collision — montrer la réponse de l'un servie à l'autre.
3. **Réparation** : clé composite `(prompt, model)` — et mesure : taux de hit avant/après, que perd-on à compter plus fin ?

**Indices :**
- `# Étape 1` : pour les stubs, s'inspirer du save/restore de l'exemple guidé 2b
- `# Étape 2` : la clé amnésique se démontre en deux appels, la collision en trois
- `# Étape 3` : réutiliser `cache.stats()` pour chiffrer ce que la clé composite coûte
- `# Étape 4` : conclure — quelles clés un cache de production doit-elle porter ?

In [16]:
# Exercice 2ter - Faire echouer le cache
# TODO etudiant
# Etape 1 : ecrire cached_completion_v2 naif (cle = hash du prompt seul) et deux
#           stubs de completion qui repondent differemment
# Etape 2 : montrer la mauvaise reponse servie (meme prompt, modeles differents)
# Etape 3 : construire une collision sur une cle faible (prompt[:8]) et la montrer
# Etape 4 : reparer avec une cle composite (prompt, model) et comparer cache.stats()

print("Exercice a completer")

Exercice a completer


### Interprétation des Résultats d'Optimisation

**Observation sur l'exécution :** la section 5 ci-dessus affiche, pour chacun
des deux appels à `optimized_completion`, les tokens d'entrée et de sortie ainsi
que le coût estimé. Comparer ces lignes montre comment la longueur de la réponse
influe sur le coût. La fonction `optimized_completion` est un simple wrapper
Chat Completions avec calcul de coût — elle **n'active pas de cache**
(`store=True` n'est pas utilisé dans cette fonction). À modèle et tarif identiques,
la différence de coût vient donc des tokens effectivement comptabilisés.

> **Note technique (issue #3571)** : `gpt-5-mini` étant un modèle de raisonnement,
> le nombre de tokens de sortie inclut les tokens de raisonnement (déduits de
> `max_completion_tokens`). Le budget est relevé à 4000 pour laisser une marge
> confortable de contenu final.

**Scénario optimal pour le cache (conceptuel) :**

Pour bénéficier du cache OpenAI (prompt caching), il faut répéter un préfixe
identique entre requêtes. Imaginons une application de support client avec un
contexte fixe :

```python
# Contexte partagé (200 tokens)
context = "Documentation produit X: [longue description]..."

# Requête 1
q1 = context + " Question: Comment l'installer?"
# → Tokens input: 200 (contexte) + 5 (question) = 205

# Requête 2 (même préfixe context) avec prompt caching activé
q2 = context + " Question: Comment le configurer?"
# → Le préfixe context est servi depuis le cache à prix réduit
```

**Points critiques (si vous activez le prompt caching) :**

1. ✅ **Répéter un préfixe identique** : Le cache ne fonctionne que sur les préfixes exacts
2. ⚠️ **Attention aux modifications** : Changer 1 mot au début du contexte invalide tout le cache
3. ⚠️ **Durée de vie** : Le cache expire après une période d'inactivité

## 6. Monitoring et Logging

Bonnes pratiques pour le monitoring en production :

### Logging Structuré
- **Timestamp** : Horodatage de chaque requête
- **Modèle** : Quel modèle a été utilisé
- **Durée** : Temps de réponse
- **Tokens** : Consommation de tokens
- **Succès/Échec** : Status de la requête
- **Erreurs** : Type et message d'erreur

### Métriques à Surveiller
- **Latence** : p50, p95, p99
- **Taux d'erreur** : Pourcentage de requêtes échouées
- **Coûts** : Dépenses quotidiennes/mensuelles
- **Tokens/requête** : Moyenne et variance
- **Rate limiting** : Nombre de requêtes rejetées

### Outils Recommandés
- **Logging** : Python `logging`, Loguru
- **APM** : Datadog, New Relic, OpenTelemetry
- **Alerting** : PagerDuty, Opsgenie
- **Dashboards** : Grafana, Kibana

In [17]:
import logging
from datetime import datetime

# Configuration du logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(name)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger("OpenAI_Production")

def logged_completion(prompt: str, model: str = None) -> str:
    """Completion avec logging complet"""
    model = model or DEFAULT_MODEL
    start = datetime.now()
    
    try:
        # Budget relevé à 4000 : gpt-5-mini est un modèle de raisonnement dont les
        # tokens de raisonnement sont déduits de max_completion_tokens. Avec 200,
        # le raisonnement pouvait consommer tout le budget sans texte final
        # visible (issue #3571).
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_completion_tokens=4000
        )
        
        duration = (datetime.now() - start).total_seconds()
        logger.info(f"SUCCESS | Model: {model} | Duration: {duration:.2f}s | "
                   f"Tokens: {response.usage.total_tokens} (in: {response.usage.prompt_tokens}, out: {response.usage.completion_tokens})")
        
        return response.choices[0].message.content
        
    except Exception as e:
        duration = (datetime.now() - start).total_seconds()
        logger.error(f"FAILED | Model: {model} | Duration: {duration:.2f}s | "
                    f"Error: {type(e).__name__}: {str(e)[:100]}")
        raise

# Tests
print("=== Test réussi ===")
result = logged_completion("Quelle heure est-il?")
print(f"Réponse: {result}\n")

print("=== Test avec modèle invalide ===")
try:
    result = logged_completion("Test", model="gpt-invalid-model")
except Exception as e:
    print(f"Erreur capturée: {type(e).__name__}")

=== Test réussi ===


2026-11-01 04:54:03 | httpx | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-11-01 04:54:03 | OpenAI_Production | INFO | SUCCESS | Model: gpt-5-mini | Duration: 6.19s | Tokens: 613 (in: 11, out: 602)


2026-11-01 04:54:03 | httpx | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 404 Not Found"


2026-11-01 04:54:03 | OpenAI_Production | ERROR | FAILED | Model: gpt-invalid-model | Duration: 0.19s | Error: NotFoundError: Error code: 404 - {'error': {'message': 'The model `gpt-invalid-model` does not exist or you do not 


Réponse: Je n’ai pas accès à l’heure en temps réel ni à l’horloge de votre appareil, donc je ne peux pas dire l’heure exacte maintenant.

Options :
- Regardez la barre d’état de votre téléphone/ordinateur ou utilisez un assistant vocal (Siri/Google/Alexa).
- Dites-moi votre ville ou votre fuseau horaire (ex. « Paris » ou « Europe/Paris ») et je vous indiquerai le décalage par rapport à l’UTC ou comment convertir une heure donnée.
- Si vous me donnez une heure de référence (par ex. l’heure affichée sur un autre appareil), je peux la convertir dans votre fuseau.

Que préférez-vous ?

=== Test avec modèle invalide ===
Erreur capturée: NotFoundError


### Exemple guidé 3 - EX-0435 (crédité #21043) : Suivi des couts avec alerte budget

L'objectif est de créer une classe `CostTracker` qui accumule les couts de chaque appel API et declenche une alerte quand un seuil est atteint.

**Indices :**
- `# Étape 1` : Créer la classe avec un attribut total_cost et un budget_max
- `# Étape 2` : Implementer track(input_tokens, output_tokens, model) qui calcule le cout et l'ajoute au total
- `# Étape 3` : Verifier si total_cost depasse budget_max et afficher un warning si oui
- `# Étape 4` : Implementer summary() qui retourne un dict avec total_cost, nb_requetes, cout_moyen

> **Exemple guidé 3 - EX-0435 (crédité #21043).** La résolution ci-dessous est celle du groupe de la PR #21043 : tarifs pédagogiques en `Decimal`, validation des entrées, alerte de budget sans blocage. Seule retouche par rapport au dépôt du groupe : la ligne de dédicace nominative du docstring a été retirée (les prénoms des étudiantes ne sont pas reproduits dans ce dépôt public), tout le reste est conservé tel quel. L'exercice EX-0435bis qui la suit rejoue ce suivi sur un budget de cours complet.


In [18]:
"""EX-0435 : tarifs pédagogiques du notebook, pas une facture fournisseur.

Préparé avec l'assistance de ChatGPT par le groupe de la PR #21043.
"""
from decimal import Decimal
import logging

class CostTracker:
    """Suivi local en USD. L'alerte ne bloque pas les appels API."""
    # Tarifs imposés par l'énoncé, en USD par million de tokens.
    PRICES = {"gpt-5-mini": (Decimal("0.15"), Decimal("0.60"))}

    def __init__(self, budget_max=1.00):
        if isinstance(budget_max, bool):
            raise ValueError("Le budget doit être un nombre positif ou nul.")
        try:
            budget = Decimal(str(budget_max))
        except Exception as exc:
            raise ValueError("Budget invalide.") from exc
        if not budget.is_finite() or budget < 0:
            raise ValueError("Le budget doit être fini et positif ou nul.")
        self.budget_max = budget
        self.total_cost = Decimal("0")
        self.nb_requetes = 0

    def track(self, input_tokens, output_tokens, model="gpt-5-mini"):
        for value in (input_tokens, output_tokens):
            if isinstance(value, bool) or not isinstance(value, int) or value < 0:
                raise ValueError("Les tokens doivent être des entiers positifs ou nuls.")
        if model not in self.PRICES:
            raise ValueError(f"Tarif non configuré pour {model!r}.")
        price_in, price_out = self.PRICES[model]
        cost = (input_tokens * price_in + output_tokens * price_out) / Decimal(1_000_000)
        self.total_cost += cost
        self.nb_requetes += 1
        if self.total_cost >= self.budget_max:
            logging.getLogger("CostTracker").warning(
                "Budget atteint ou dépassé : %s USD / %s USD",
                self.total_cost, self.budget_max,
            )
        return cost

    def summary(self):
        average = self.total_cost / self.nb_requetes if self.nb_requetes else Decimal("0")
        return {"total_cost": float(self.total_cost),
                "nb_requetes": self.nb_requetes,
                "avg_cost_per_request": float(average)}

def demo():
    logging.basicConfig(level=logging.WARNING, format="%(levelname)s : %(message)s")
    print("EX-0435 | Simulation locale, aucun appel API")
    print("Tarifs pédagogiques : 0.15 USD / 1M entrée, 0.60 USD / 1M sortie")
    tracker = CostTracker(budget_max="0.01")
    for number, (tokens_in, tokens_out) in enumerate([(100, 50), (200, 100), (10000, 15000)], 1):
        cost = tracker.track(tokens_in, tokens_out)
        print(f"Appel {number} : coût = {cost:.6f} USD, cumul = {tracker.total_cost:.6f} USD")
    print("Bilan :", tracker.summary())
    print("Le troisième appel dépasse 0.01 USD. Le warning ne bloque aucun appel.")


# Démonstration autonome de la correction EX-0435.
demo()


2026-11-01 04:54:03 | CostTracker | WARNING | Budget atteint ou dépassé : 0.010635 USD / 0.01 USD


EX-0435 | Simulation locale, aucun appel API
Tarifs pédagogiques : 0.15 USD / 1M entrée, 0.60 USD / 1M sortie
Appel 1 : coût = 0.000045 USD, cumul = 0.000045 USD
Appel 2 : coût = 0.000090 USD, cumul = 0.000135 USD
Appel 3 : coût = 0.010500 USD, cumul = 0.010635 USD
Bilan : {'total_cost': 0.010635, 'nb_requetes': 3, 'avg_cost_per_request': 0.003545}
Le troisième appel dépasse 0.01 USD. Le warning ne bloque aucun appel.


### EXERCICE EX-0435bis - Le budget du cours : quand l'alerte tombe

L'exemple guidé 3 suit les coûts appel par appel. Reste la question du gestionnaire : sur un **plan de charge** connu, à quelle requête l'alerte se déclenche-t-elle, et quelle marge reste-t-il ?

**Objectif** : rejouer le `CostTracker` de l'exemple guidé (tarifs gpt-5-mini : 0,15 USD/M en entrée, 0,60 USD/M en sortie) sur le plan de charge ci-dessous, budget 1,00 USD, et localiser le passage à l'alerte.

Plan de charge (input_tokens, output_tokens) — le profil d'une journée de cours, répété sur quatre semaines :

```python
JOUR = [(12000, 3500), (8000, 4200), (15000, 5000), (6000, 2800),
        (20000, 6500), (9000, 3100), (11000, 4000), (25000, 7000)]
PLAN = JOUR * 28  # 224 requêtes au total
```

**Indices** :
- `# Étape 1` : instancier `CostTracker(budget_max=1.00)` et parcourir `PLAN` avec `track(...)`
- `# Étape 2` : relever `cout_total_usd` (arrondi à 4 décimales) et `requete_declenchant_alerte` — le numéro (1-indexé) de la requête du plan à partir de laquelle `total_cost >= budget_max` est signalé
- `# Étape 3` : calculer `marge_restante` (Decimal) — l'écart au terme du plan entre `cout_total_usd` et le budget (positif s'il est dépassé, négatif sinon)
- `# Indice` : restez en `Decimal` du début à la fin ; un float arrondi trop tôt décale le numéro de la requête frontière

In [19]:
# EX-0435bis : le budget du cours -- quand l'alerte tombe
# TODO etudiant : rejouer CostTracker sur PLAN, budget 1.00 USD
# Etape 1 : parcourir PLAN avec track(input, output)
# Etape 2 : relever requete_declenchant_alerte (1-indexee) et cout_total_usd
# Etape 3 : calculer marge_restante (Decimal, jamais float)
# Indice : l'alerte du tracker est informative, elle n'arrete pas la boucle
JOUR = [(12000, 3500), (8000, 4200), (15000, 5000), (6000, 2800),
        (20000, 6500), (9000, 3100), (11000, 4000), (25000, 7000)]
PLAN = JOUR * 28  # 224 requêtes : quatre semaines de même profil
cout_total_usd = None  # TODO etudiant
requete_declenchant_alerte = None  # TODO etudiant
marge_restante = None  # TODO etudiant
print("EX-0435bis a completer : budget du cours")

EX-0435bis a completer : budget du cours


### Interprétation des Logs de Production

**Analyse des logs générés :** le journal de la cellule de monitoring ci-dessus
est la seule référence pour les horodatages, durées et comptes de tokens de cette
exécution. Il comporte une ligne `SUCCESS` pour le modèle configuré et une ligne
`FAILED` pour le modèle volontairement invalide.

**Points clés observés :**

1. **Latence** :
   - La ligne `SUCCESS` donne la durée end-to-end de la requête aboutie.
   - La ligne `FAILED` donne celle de l'échec ; comparer les deux renseigne sur
     le coût temporel d'une erreur, sans confondre cette mesure avec un timeout.

2. **Consommation de tokens** :
   - La ligne `SUCCESS` sépare le total, l'entrée et la sortie ; ces trois comptes
     décrivent uniquement la requête aboutie.
   - Le coût se calcule en appliquant séparément les tarifs d'entrée et de sortie,
     comme le fait `optimized_completion`, plutôt qu'en figeant un montant dans
     cette lecture.

3. **Gestion d'erreur** :
   - Exception capturée et loggée (pas de crash)
   - Message d'erreur informatif : "The model `gpt-invalid-model` does not exist"
   - L'application peut réagir (fallback, alerte utilisateur)

**Métriques à extraire pour dashboards :**

| Métrique | Calcul | Seuil alerte |
|----------|--------|--------------|
| **Latence P95** | 95ème percentile des durées | > 3s |
| **Taux d'erreur** | (erreurs / total) × 100 | > 5% |
| **Coût moyen/requête** | Somme(tokens × prix) / nb_requêtes | > $0.01 |
| **Requêtes/minute** | Count(logs) sur fenêtre 1min | > 80% de la limite |

> **Production tip** : Exporter les logs vers un système centralisé (ELK, Datadog) pour analyser les tendances sur plusieurs jours et détecter les anomalies (ex: latence soudainement × 3).

## 7. Patterns Avancés : Streaming et Modération

### Streaming pour UX Réactive

Le streaming permet d'afficher la réponse progressivement :
- **Meilleure UX** : L'utilisateur voit la réponse se construire
- **Latence perçue réduite** : Premier token en ~200ms vs 5s pour réponse complète
- **Annulation précoce** : Possibilité de stopper si réponse non pertinente

### Modération de Contenu

OpenAI propose une API de modération pour détecter :
- Contenu haineux/violent
- Harcèlement
- Contenu sexuel
- Auto-mutilation
- Etc.

**Pattern recommandé :**
1. Modérer l'input utilisateur
2. Générer la réponse si OK
3. Modérer l'output avant affichage

In [20]:
# Streaming
print("=== Streaming Example ===")
# Budget relevé à 4000 : gpt-5-mini est un modèle de raisonnement dont les
# tokens de raisonnement sont déduits de max_completion_tokens. Avec 100,
# le raisonnement pouvait consommer tout le budget sans texte final visible
# (issue #3571).
stream = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": "Écris un haïku sur la programmation."}],
    stream=True,
    max_completion_tokens=4000
)

print("Réponse streamée: ", end="", flush=True)
for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print("\n")

# Modération
print("=== Moderation Example ===")
test_inputs = [
    "Bonjour, comment vas-tu?",
    "Enfoiré, je vais te tuer!"
]

for text in test_inputs:
    moderation = client.moderations.create(input=text)
    result = moderation.results[0]
    
    print(f"\nTexte: {text}")
    print(f"Flaggé: {result.flagged}")
    if result.flagged:
        print(f"Catégories: {[cat for cat, flagged in result.categories.__dict__.items() if flagged]}")

=== Streaming Example ===


2026-11-01 04:54:38 | httpx | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Réponse streamée: 

Cl

avier

...



=== Moderation Example ===


2026-11-01 04:54:38 | httpx | INFO | HTTP Request: POST https://api.openai.com/v1/moderations "HTTP/1.1 200 OK"



Texte: Bonjour, comment vas-tu?
Flaggé: False


2026-11-01 04:54:38 | httpx | INFO | HTTP Request: POST https://api.openai.com/v1/moderations "HTTP/1.1 200 OK"



Texte: Enfoiré, je vais te tuer!
Flaggé: True
Catégories: ['harassment', 'harassment_threatening', 'violence']


### Interprétation des Résultats Streaming et Modération

**Observations sur le Streaming :**

Le streaming transforme radicalement l'expérience utilisateur :
- **Latence perçue** : Au lieu d'attendre 2-3s pour voir la réponse complète, le premier mot apparaît en ~200ms
- **Engagement utilisateur** : Voir le texte se construire donne l'impression d'une "conversation naturelle"
- **Format de réponse** : Les chunks arrivent progressivement via `delta.content`

**Observations sur la Modération :**

| Texte | Flaggé | Catégories détectées | Action recommandée |
|-------|--------|---------------------|-------------------|
| "Bonjour, comment vas-tu?" | ❌ Non | Aucune | ✅ Traiter normalement |
| "Enfoiré, je vais te tuer!" | ✅ Oui | harassment, harassment_threatening, violence | 🚫 Bloquer avant génération |

**Pattern de sécurité production :**

```python
# Workflow complet sécurisé
def safe_generation(user_input: str) -> str:
    # 1. Modération input
    mod_input = client.moderations.create(input=user_input)
    if mod_input.results[0].flagged:
        return "Votre message viole notre politique d'utilisation."
    
    # 2. Génération
    response = optimized_completion(user_input)["content"]
    
    # 3. Modération output
    mod_output = client.moderations.create(input=response)
    if mod_output.results[0].flagged:
        return "Désolé, impossible de générer une réponse appropriée."
    
    return response
```

> **Note importante** : La modération ajoute ~100ms de latence par check. Pour les applications temps-réel, envisager une modération asynchrone post-génération.

## 8. Checklist Déploiement Production

### Sécurité
- [ ] API keys stockées dans variables d'environnement (jamais hardcodées)
- [ ] Rotation régulière des clés
- [ ] Rate limiting côté serveur
- [ ] Validation et sanitization des inputs utilisateur
- [ ] Modération de contenu activée
- [ ] Logs ne contiennent pas de données sensibles

### Résilience
- [ ] Retry automatique avec backoff exponentiel
- [ ] Timeout configurés
- [ ] Circuit breaker pour dépendances externes
- [ ] Fallback gracieux en cas d'erreur
- [ ] Health checks réguliers

### Performance
- [ ] Cache activé (`store=True`)
- [ ] Choix du modèle adapté au cas d'usage
- [ ] `max_tokens` configuré pour limiter les coûts
- [ ] Streaming pour réponses longues
- [ ] Background mode pour tâches longues

### Monitoring
- [ ] Logging structuré configuré
- [ ] Métriques : latence, taux d'erreur, coûts
- [ ] Alertes budgétaires configurées
- [ ] Dashboards temps réel
- [ ] Traçabilité des requêtes (request ID)

### Coûts
- [ ] Budget mensuel défini
- [ ] Alertes à 50%, 80%, 100% du budget
- [ ] Audit régulier de l'usage par endpoint/utilisateur
- [ ] Optimisation continue des prompts
- [ ] Évaluation régulière des modèles (nouveaux modèles moins chers?)

### Conformité
- [ ] RGPD : consentement utilisateur pour traitement des données
- [ ] Politique de rétention des données
- [ ] Anonymisation des données personnelles
- [ ] Documentation des traitements de données
- [ ] DPO informé de l'usage d'IA

## Conclusion

### Récapitulatif des Patterns

| Pattern | Cas d'usage | Complexité |
|---------|-------------|------------|
| **Responses API + store** | Conversations courtes avec cache | ⭐⭐ |
| **Conversations API** | Conversations longue durée | ⭐⭐ |
| **Background Mode** | Tâches longues (>30s) | ⭐⭐⭐ |
| **Retry + Backoff** | Résilience production | ⭐⭐ |
| **Rate Limiter** | Respect limites API | ⭐⭐ |
| **Logging structuré** | Monitoring et debug | ⭐⭐ |
| **Streaming** | UX temps réel | ⭐⭐ |
| **Modération** | Sécurité contenu | ⭐ |

### Ressources Supplémentaires

**Documentation OpenAI :**
- [Responses API](https://platform.openai.com/docs/api-reference/responses)
- [Conversations API](https://platform.openai.com/docs/api-reference/conversations)
- [Rate Limits](https://platform.openai.com/docs/guides/rate-limits)
- [Error Codes](https://platform.openai.com/docs/guides/error-codes)

**Bibliothèques Python :**
- [tenacity](https://tenacity.readthedocs.io/) : Retry robuste
- [openai](https://github.com/openai/openai-python) : SDK officiel
- [loguru](https://loguru.readthedocs.io/) : Logging simplifié

**Prochaines étapes :**
- Implémenter ces patterns dans votre application
- Configurer un monitoring complet
- Tester la résilience (chaos engineering)
- Optimiser les coûts progressivement

## Exercices Pratiques

### Exercice 1 : Chatbot Multi-Session (30 min)

Créez un chatbot de support client qui :
1. Utilise la Conversations API pour maintenir le contexte
2. Persiste les conversations dans un fichier JSON (pour reprise après redémarrage)
3. Implémente un retry avec backoff
4. Log toutes les interactions

**Bonus :** Ajouter une commande `/summary` qui résume la conversation en cours.

### Exercice 2 : Système d'Analyse de Documents (40 min)

Implémentez un système qui :
1. Prend un long document en entrée
2. Lance l'analyse en background mode
3. Affiche une barre de progression pendant le traitement
4. Retourne un rapport structuré (points clés, résumé, recommandations)
5. Calcule et affiche le coût total de l'analyse

**Bonus :** Comparer les coûts entre gpt-5-mini et gpt-5.

### Exercice 3 : Rate Limiter Multi-Tier (20 min)

Améliorez la classe `RateLimiter` pour supporter :
1. Des limites par minute ET par jour
2. Des priorités de requêtes (high/medium/low)
3. Un mode "burst" (permettre 10 requêtes instantanées, puis throttling)

**Bonus :** Ajouter des statistiques (nombre de requêtes dans les dernières 24h, temps moyen d'attente).